In [1]:
# home llms, office olm2
import warnings
warnings.filterwarnings("ignore", category=SyntaxWarning, message=".*escape sequence.*")

import httpx
from openai import OpenAI
import time
import pandas as pd
import numpy as np
import os
from dotenv import load_dotenv, find_dotenv
from statistics import mode
import ast
import re

#from aux_prompts import *
#from queries import criteria_text3
os.environ["OPENAI_API_KEY"] = 'hey' # for importing trialmind, api_key should be set beforehand
os.environ['TEMPERATURE'] = '0'

use = 'rogozin'
#use = 'cpp'

if use == 'hf':
    os.environ["BASE_URL"] = "https://router.huggingface.co/v1"
    os.environ["MODEL_NAME"] ='Qwen/Qwen3-14B'
    os.environ["OPENAI_API_KEY"] = os.getenv("HUGGINGFACE_HUB_TOKEN")
elif use == 'olm':
    os.environ["BASE_URL"] = "http://localhost:11434/v1"
    os.environ["MODEL_NAME"] ='qwen3:14b'
    os.environ["OPENAI_API_KEY"] = 'hey'
elif use == 'rogozin':
    os.environ["BASE_URL"] = "http://localhost:8501/v1"
    os.environ["MODEL_NAME"] ='/models/Qwen3.6-27B-UD-Q4_K_XL.gguf'#'ggml-org/gemma-3-1b-it-GGUF'
    os.environ["TRANSL_MODEL_NAME"] = '/models/Qwen3.6-27B-UD-Q4_K_XL.gguf'
    os.environ["OPENAI_API_KEY"] = 'hello'
elif use == 'sirius':
    os.environ["BASE_URL"] = "http://127.0.0.1:8082/v1"
    os.environ["MODEL_NAME"] ='Qwen3.6-27B'#'ggml-org/gemma-3-1b-it-GGUF'
    os.environ["OPENAI_API_KEY"] = 'hello'
    os.environ["TRANSL_MODEL_NAME"] = 'Qwen3.6-27B'
elif use == 'cloud':
    os.environ["BASE_URL"] = "http://51.250.66.187:8080/v1"
    os.environ["MODEL_NAME"] ='Qwen3.6-27B'
    #os.environ["TRANSL_MODEL_NAME"] = 'gemma-4-12B'
    os.environ["OPENAI_API_KEY"] = 'hey'
load_dotenv(find_dotenv(usecwd=True), override=True)
import glob
import main

%load_ext autoreload
%autoreload 2

c:\Users\MKoshkareva\miniconda3\envs\my_project\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


### small fixes

In [22]:
dfo = pd.read_csv('new_criteria_ana_en.csv')
dfo.PMID = dfo.PMID.apply(lambda x: str(x.split('/')[-1]))
print(dfo.shape)

dfe = pd.read_csv('common_errors_fixed.csv')
dfe['PMID_статьи'] = dfe['PMID_статьи'].apply(lambda x: str(x).split('.')[0])

def fix(x):
    if x['PMID'] in dfe['PMID_статьи'].values:
        row = dfe[dfe['PMID_статьи'] == x['PMID']]
        row_1 =  row[row['Номер_критерия']==1]
        row_2 =  row[row['Номер_критерия']==2]
        row_3 =  row[row['Номер_критерия']==3]
        for row_i in [row_1,row_2,row_3]:
            if row_i.shape[0]:
                i = str(row_i['Номер_критерия'].values[0]).split('.')[0]
                x[f's{i}'] = row_i['Оценка_Эксперт'].values[0]
                x[f'r{i}'] = row_i['Обоснование_Эксперт'].values[0]
                print('change', x['PMID'])
        return x
    else:
        return x

dfo = dfo.apply(lambda x: fix(x), axis=1)
ps_thresh = [1,1,0]
chosen_df = dfo[(dfo['s1']>=ps_thresh[0]
                    )&(dfo['s2']>=ps_thresh[1]
                        )&(dfo['s3']>=ps_thresh[2])]
dfo['accept'] = dfo[['s1','s2','s3']].apply(lambda x: True if (x['s1']>=ps_thresh[0]
                    )&(x['s2']>=ps_thresh[1]
                        )&(x['s3']>=ps_thresh[2]) else False, axis=1)

dfo.accept.value_counts()

(67, 7)
change 25139705
change 31399568
change 26995877
change 35343863
change 35343863
change 26438251
change 26438251
change 37773318
change 37773318
change 37773318
change 33676682
change 23719267
change 23719267
change 22845480
change 24789362
change 37686423
change 41213063
change 41213063
change 36801912
change 28465216
change 28465216
change 36129915
change 36627781
change 36627781
change 34898475
change 34898475
change 28903995
change 28903995
change 23585220
change 26468446
change 23656699
change 29466887
change 40526090
change 40526090
change 38378070


accept
False    40
True     27
Name: count, dtype: int64

In [23]:
from googletrans import SyncTranslator
import re
translator = SyncTranslator()
text = (dfo.iloc[:,4:-1].values).reshape(-1)
text = [re.sub(r'\b[нН][мМ][Рр][лЛ]\b','NSCLC', re.sub(r'\b[Кк][Рр][Рр]\b','CRC',i)) for i in text]
r = translator.translate([i for i in text], dest="en")
[i.text for i in r]#.text

ImportError: cannot import name 'SyncTranslator' from 'googletrans' (c:\Users\MKoshkareva\miniconda3\envs\my_project\Lib\site-packages\googletrans\__init__.py)

In [ ]:
dfo.iloc[:,4:-1] = np.array([i.text for i in r]).reshape(-1,3)
#dfo.to_csv('new_criteria_ana_en2.csv',index=False)

In [ ]:
dfo.iloc[:,4:-1].values

array([['The article describes the clinical case of a 19-year-old patient with malignant peritoneal mesothelioma',
        'Crizotinib has been used to treat mesothelioma',
        'The effectiveness of crizotinib for CRC has not been assessed.'],
       ['"16 colorectal cancer cases"',
        'The abstract states that patients received inhibitors of various signaling pathways, including ALK/ROS1. Although crizotinib is an ALK/ROS1 inhibitor, it is not explicitly named in the abstract. We cannot confirm that it was the specific drug used in patients (it could be ceritinib, alectinib, lorlatinib, etc.)',
        'The abstract provides general data for the cohort (4 patients with CRC had a response), but is not linked to a specific drug.'],
       ['This is a review article summarizing current advances in targeted therapy for solid tumors. It does not include original patients, models or samples',
        'Crizotinib is mentioned in the abstract, but in association with NSCLC',
        

### back

In [ ]:
dfo = pd.read_csv('new_criteria_ana_en2.csv')
dfo['PMID'] = dfo['PMID'].astype(str)
dfo[['s1','s2','s3']] = dfo[['s1','s2','s3']].astype(int)
dfo.head(2)

,PMID,s1,s2,s3,r1,r2,r3,accept
0,34766064,-1,-1,-1,The article describes the clinical case of a 1...,Crizotinib has been used to treat mesothelioma,The effectiveness of crizotinib for CRC has no...,False
1,32208873,1,-1,-1,"""16 colorectal cancer cases""",The abstract states that patients received inh...,The abstract provides general data for the coh...,False


In [ ]:
from sklearn.model_selection import train_test_split

X = dfo.iloc[:,:1]
y = dfo.iloc[:,]
# stratify by accept/not accept. maybe instead by [1,1,0]?
X_train, X_test, y_train, y_test = train_test_split(X,y, train_size=0.7, 
                                                    random_state=42,stratify=dfo.accept)

In [ ]:
df_p_e = main.run_screening(pmid_list=X_train.PMID.values, 
                            fin_condition='Colorectal cancer', 
                            treatment='Crizotinib',
                            ps_thresh = [1,1,0], real=False)
df_p_e.to_csv('screening.csv',index=False)


INFO:root:COLLECTING PAPER TEXTS
INFO:root:1.0490188598632812
INFO:root:SCREENING PAPERS
INFO:root:0.008991241455078125


In [ ]:
df_p_e.head(2)

,PMID,s1,s2,s3,r1,r2,r3,Fin_text
0,34898475,1,1,0,The study explicitly focuses on 'colorectal ca...,The results section explicitly mentions 'S-cri...,The study identifies Crizotinib via 'in silico...,Identification of a novel immune signature for...
1,28320945,1,1,1,The study explicitly uses a 'human colorectal ...,The text states: 'Addition of the dual MET/RON...,The study evaluates therapeutic effectiveness ...,Three-dimensional culture system identifies a ...


In [ ]:
y_train.head(2)

,PMID,s1,s2,s3,r1,r2,r3,accept
41,34898475,1,-1,-1,"""expression profiles and clinical characterist...",The drug is mathematically predicted to be pot...,Therapeutic efficacy has not been assessed,False
8,28320945,1,1,1,"""human colorectal cancer (CRC) cell line (HCA-...","""Addition of the dual MET/RON tyrosine kinase ...",Evaluates the therapeutic efficacy of crizotin...,True


In [ ]:
df_p_e = pd.read_csv('screening.csv')
errors_df, correct_df = main.find_mistakes(df_p_e, y_train)
print(errors_df.shape)
errors_df.head(3)

(33, 7)


,PMID,num,Fin_text,s_pred,r_pred,s_expert,r_expert
3,22845480,1,Personalized medicine and pharmacogenetic biom...,1,The text explicitly mentions 'KRAS against the...,-1,"CRC is mentioned once in the list, but only in..."
12,23585220,1,Symptomatic reduction in free testosterone lev...,0,The provided text mentions 'male cancer patien...,-1,"The study includes men with cancer, but there ..."
13,23585220,2,Symptomatic reduction in free testosterone lev...,0,"While the study investigates 'Crizotinib', the...",-1,The drug is not being studied specifically for...


In [ ]:
#error_id_df = main.find_ind_errors(errors_df=errors_df,
                            fin_condition='Colorectal cancer', 
                            treatment='Crizotinib')
print(error_id_df.shape)
error_id_df.to_csv('outputs/error_id_df.csv', index=False)
error_id_df.head(2)

INFO:root:22845480, 1, The text explicitly mentions 'KRAS against the use of cetuximab and panitumumab in colorectal cancer', confirming that colorectal cancer is a studied disease/condition in the context of the biomarkers
INFO:root:23585220, 1, The provided text mentions 'male cancer patients' but does not specify the type of cancer (e.g., colorectal, lung, etc.). Therefore, it is uncertain if the study includes patients with colorectal canc
INFO:root:23585220, 2, While the study investigates 'Crizotinib', the text does not specify the underlying cancer type for which the patients were being treated. Without confirmation that the patients have colorectal cancer
INFO:root:23585220, 3, The study evaluates symptomatic reduction in testosterone levels and hypogonadal effects, not the therapeutic effectiveness or antitumor activity of Crizotinib against the cancer itself. Furthermore, 
INFO:root:23719267, 3, The text discusses the promise of crizotinib in NSCLC but does not provide specif

http://localhost:8501/v1 /models/Qwen3.6-27B-UD-Q4_K_XL.gguf
http://localhost:8501/v1 /models/Qwen3.6-27B-UD-Q4_K_XL.gguf
http://localhost:8501/v1 /models/Qwen3.6-27B-UD-Q4_K_XL.gguf
http://localhost:8501/v1 /models/Qwen3.6-27B-UD-Q4_K_XL.gguf
http://localhost:8501/v1 /models/Qwen3.6-27B-UD-Q4_K_XL.gguf
http://localhost:8501/v1 /models/Qwen3.6-27B-UD-Q4_K_XL.gguf
http://localhost:8501/v1 /models/Qwen3.6-27B-UD-Q4_K_XL.gguf
http://localhost:8501/v1 /models/Qwen3.6-27B-UD-Q4_K_XL.gguf
http://localhost:8501/v1 /models/Qwen3.6-27B-UD-Q4_K_XL.gguf
http://localhost:8501/v1 /models/Qwen3.6-27B-UD-Q4_K_XL.gguf
http://localhost:8501/v1 /models/Qwen3.6-27B-UD-Q4_K_XL.gguf
http://localhost:8501/v1 /models/Qwen3.6-27B-UD-Q4_K_XL.gguf
http://localhost:8501/v1 /models/Qwen3.6-27B-UD-Q4_K_XL.gguf
http://localhost:8501/v1 /models/Qwen3.6-27B-UD-Q4_K_XL.gguf
http://localhost:8501/v1 /models/Qwen3.6-27B-UD-Q4_K_XL.gguf
http://localhost:8501/v1 /models/Qwen3.6-27B-UD-Q4_K_XL.gguf
http://localhost:8501/v1

INFO:root:fresh response in asynch: reasoning='The model incorrectly interpreted the mention of colorectal cancer as a responder subgroup as sufficient evidence for inclusion. The expert correctly identified that while CRC patients were present, the study lacked specific focus, detailed analysis, or dedicated models for CRC, making the inclusion status uncertain rather than confirmed.' error="The model erroneously classified the study as including CRC patients based solely on a list of responder types, ignoring the expert's point that CRC was not a focused cohort with specific analysis or models, rendering the inclusion uncertain."
INFO:root:
INFO:root:fresh response in asynch: reasoning='The model incorrectly interpreted the mention of colorectal cancer as evidence that the disease itself was the subject of the study. The abstract is a review of pharmacogenetic biomarkers across various cancers. Colorectal cancer is merely cited as an example context for the KRAS biomarker, not as the

(33, 9)


,PMID,num,Fin_text,s_pred,r_pred,s_expert,r_expert,error_id_reason,error_id
3,22845480,1,Personalized medicine and pharmacogenetic biom...,1,The text explicitly mentions 'KRAS against the...,-1,"CRC is mentioned once in the list, but only in...",The model incorrectly interpreted the mention ...,The model failed to distinguish between the st...
12,23585220,1,Symptomatic reduction in free testosterone lev...,0,The provided text mentions 'male cancer patien...,-1,"The study includes men with cancer, but there ...",The model assigned a score of 0 (UNCERTAIN) be...,The model incorrectly scored UNCERTAIN (0) ins...


In [ ]:
# TO COMPARE WITH EXPERT
q = error_id_df[['PMID','num','error_id']]


df1 = pd.read_csv('common_errors_fixed.csv')
for i in ['PMID_статьи','Номер_критерия','Оценка_ЛЛМ','Оценка_Эксперт']:
    if i =='PMID_статьи':
        df1[i] = df1[i].apply(lambda x: str(int(x)))
    else:
        df1[i] = df1[i].apply(lambda x: int(x))
df1 = df1[['PMID_статьи','Номер_критерия',
    'Почему_ошибка','Тип_ошибки']]
print(df1.shape)
df1.head(2)

NameError: name 'error_id_df' is not defined

In [ ]:
merged = pd.merge(q,df1,left_on=['PMID','num'],right_on=['PMID_статьи','Номер_критерия'],
        how='left')
print(merged.shape)
merged.drop(columns=['PMID_статьи','Номер_критерия']).values

ValueError: You are trying to merge on int64 and object columns for key 'PMID'. If you wish to proceed you should use pd.concat

In [ ]:
fin_condition='Colorectal cancer'
treatment='Crizotinib'

In [ ]:
fin_condition='Colorectal cancer'
treatment='Crizotinib'
error_id_df = pd.read_csv('outputs/error_id_df.csv')
error_type_df, outs = main.find_error_clusters(error_id_df,
                                                fin_condition, treatment)
print(error_type_df.shape)

error_type_df.to_csv('outputs/error_type_df.csv')
error_type_df.head(2)

INFO:root: passing 1 queries to llama.cpp...



    eligibility_criteria = ["Does the study include patients, animal models, organoids, cell lines, or biological samples representing 'Colorectal cancer' as a studied disease/condition?", "Does the study investigate 'Crizotinib' as an intervention, treatment, exposure, or experimental factor for 'Colorectal cancer'?", "Does the study evaluate the therapeutic effectiveness or antitumor activity of 'Crizotinib' for 'Colorectal cancer' using clinically or biologically relevant efficacy outcomes?"]
    model_errors = error_index | criteria_num | error_description
0 | 1 | The model failed to distinguish between the study's primary subject (biomarkers) and the diseases listed as examples. It incorrectly assumed that mentioning CRC in a list of biomarker applications meant CRC was the studied condition, ignoring that the paper reviews biomarkers generally.
1 | 1 | The model incorrectly scored UNCERTAIN (0) instead of NO (-1). It failed to recognize that the complete absence of any mention o

INFO:root:fresh response in asynch: reasoning="The errors can be clustered into four distinct root causes based on the cognitive mechanism of the failure:\n1. **Confusing Background/Example Mentions with Study Focus**: The model incorrectly interprets a disease mentioned in a list, background, or as an example as the primary subject of the study. This applies to Criteria 1 and 2.\n2. **Misinterpreting Study Design/Methodology**: The model fails to distinguish between primary studies (patients/samples) and reviews/policy analyses, or between experimental/clinical investigations and in silico/computational predictions. This applies to Criteria 1 and 2.\n3. **Treating Explicit Exclusion or Absence as Ambiguity**: The model assigns 'Uncertain' when the text explicitly excludes the condition, limits the scope to another disease, or completely lacks the required information, which should result in a 'No'. This applies to Criteria 1, 2, and 3.\n4. **Ignoring Population Specificity in Efficacy

(33, 11)


,PMID,num,Fin_text,s_pred,r_pred,s_expert,r_expert,error_id_reason,error_id,error_type_reason,error_type
0,22845480,1,Personalized medicine and pharmacogenetic biom...,1,The text explicitly mentions 'KRAS against the...,-1,"CRC is mentioned once in the list, but only in...",The model incorrectly interpreted the mention ...,The model failed to distinguish between the st...,Model confused a list of biomarker application...,Confusing background/example mentions with stu...
1,23585220,1,Symptomatic reduction in free testosterone lev...,0,The provided text mentions 'male cancer patien...,-1,"The study includes men with cancer, but there ...",The model assigned a score of 0 (UNCERTAIN) be...,The model incorrectly scored UNCERTAIN (0) ins...,Model treated complete absence of mention as a...,Treating explicit exclusion or absence as ambi...


In [ ]:
for c in outs.clusters:
    print('\n',c.name)
    print(c.notable_pointers)


 Confusing background/example mentions with study focus
['CRC listed as example of biomarker application', 'CRC mentioned in list of ROS1/ALK associated diseases', 'CRC cited illustratively in general review', 'Background mention mistaken for primary inclusion']

 Misinterpreting study design or methodology type
['Review article mistaken for primary study with patients', 'Policy/economic analysis mistaken for patient inclusion', 'In silico prediction mistaken for experimental intervention', 'Computational screening mistaken for efficacy evaluation']

 Treating explicit exclusion or absence as ambiguity
['Complete absence of CRC mention treated as uncertain', 'Explicit focus on NSCLC treated as uncertain for CRC', 'Explicit exclusion of CRC from indications treated as uncertain', 'Unspecified cancer type treated as uncertain instead of no']

 Ignoring population specificity in efficacy evaluation
['Efficacy reported for advanced solid tumors, not CRC', 'Response rates applied to other 

In [ ]:
error_type_df.error_type.value_counts()

error_type
Treating explicit exclusion or absence as ambiguity       13
Confusing background/example mentions with study focus    10
Misinterpreting study design or methodology type           7
Ignoring population specificity in efficacy evaluation     3
Name: count, dtype: int64

In [ ]:
for i in error_type_df.error_type.unique():
    print('\n',i)
    q = error_type_df[error_type_df.error_type==i]
    print(q.error_id.values)
    #error_type_df


 Confusing background/example mentions with study focus
["The model failed to distinguish between the study's primary subject (biomarkers) and the diseases listed as examples. It incorrectly assumed that mentioning CRC in a list of biomarker applications meant CRC was the studied condition, ignoring that the paper reviews biomarkers generally."
 'The model incorrectly assumed that listing colorectal cancer among various cancers with ROS1 rearrangements meant the study specifically included CRC patients or samples, failing to recognize it was just a general review of ROS1 across multiple cancer types.'
 'The model mistook a background mention of colorectal cancer in a list of ALK-associated diseases for actual study inclusion, ignoring that the research focuses on NSCLC and general mechanisms without specific CRC data.'
 "The model incorrectly classified a background example of colorectal cancer as the study's focus, failing to recognize that the abstract is a general review where CRC 

In [ ]:
error_type_df.PMID = error_type_df.PMID.astype(str)
df1 = pd.read_csv('common_errors_fixed.csv')
for i in ['PMID_статьи','Номер_критерия','Оценка_ЛЛМ','Оценка_Эксперт']:
    if i =='PMID_статьи':
        df1[i] = df1[i].apply(lambda x: str(int(x)))
    else:
        df1[i] = df1[i].apply(lambda x: int(x))
df1 = df1[['PMID_статьи','Номер_критерия',
    'Почему_ошибка','Тип_ошибки']]
    
merged = pd.merge(error_type_df[['PMID','num','error_type_reason','error_type']],
                df1,
                left_on=['PMID','num'],right_on=['PMID_статьи','Номер_критерия'],
                        how='left')
print(merged.shape)
for i in merged.drop(columns=['PMID_статьи','Номер_критерия','Почему_ошибка']).values:
    print(i[:2])
    print(f"LLM: {i[3]}; \nReason: {i[2]}")
    print(f"Expert: {i[4]}\n")

(33, 8)
['22845480' 1]
LLM: Confusing background/contextual mention with study focus or inclusion; 
Reason: The model confused a list of biomarker applications (background) with the study's primary subject.
Expert: The model confuses background keywords with the paper's focus

['23585220' 1]
LLM: Misinterpreting absence of data as ambiguity instead of definitive exclusion; 
Reason: The model treated the absence of CRC mention as ambiguous rather than definitive exclusion.
Expert: nan

['23585220' 2]
LLM: Misinterpreting absence of data as ambiguity instead of definitive exclusion; 
Reason: The model treated the absence of CRC in the population as ambiguous rather than definitive exclusion
Expert: nan

['23585220' 3]
LLM: Misinterpreting absence of data as ambiguity instead of definitive exclusion; 
Reason: The model treated the focus on side effects (excluding efficacy) as ambiguous rather than definitive
Expert: The model treats 'not mentioned' as 'possible'

['23719267' 3]
LLM: Misin

In [ ]:
outs

ErrorsAndClusters(reasoning="The errors can be clustered into four distinct root causes based on the cognitive mechanism of the failure:\n1. **Confusing Background/Example Mentions with Study Focus**: The model incorrectly interprets a disease mentioned in a list, background, or as an example as the primary subject of the study. This applies to Criteria 1 and 2.\n2. **Misinterpreting Study Design/Methodology**: The model fails to distinguish between primary studies (patients/samples) and reviews/policy analyses, or between experimental/clinical investigations and in silico/computational predictions. This applies to Criteria 1 and 2.\n3. **Treating Explicit Exclusion or Absence as Ambiguity**: The model assigns 'Uncertain' when the text explicitly excludes the condition, limits the scope to another disease, or completely lacks the required information, which should result in a 'No'. This applies to Criteria 1, 2, and 3.\n4. **Ignoring Population Specificity in Efficacy Evaluation**: The

In [2]:
error_type_df = pd.read_csv('outputs/error_type_df.csv', index_col=0)
df_p_e = pd.read_csv('screening.csv')

from sklearn.model_selection import train_test_split
dfo = pd.read_csv('new_criteria_ana_en2.csv')
dfo['PMID'] = dfo['PMID'].astype(str)
dfo[['s1','s2','s3']] = dfo[['s1','s2','s3']].astype(int)
X = dfo.iloc[:,:1]
y = dfo.iloc[:,]
# stratify by accept/not accept. maybe instead by [1,1,0]?
X_train, X_test, y_train, y_test = train_test_split(X,y, train_size=0.7, 
                                                    random_state=42,stratify=dfo.accept)

errors_df, correct_df = main.find_mistakes(df_p_e, y_train)

a = correct_df.groupby('PMID').count()['num']
a[a==3].index.values


fin_condition='Colorectal cancer'
treatment='Crizotinib'
error_id_df = pd.read_csv('outputs/error_id_df.csv')
error_type_df, outs = main.find_error_clusters(error_id_df,
                                                fin_condition, treatment)



INFO:root: passing 1 queries to llama.cpp...



    eligibility_criteria = ["Does the study include patients, animal models, organoids, cell lines, or biological samples representing 'Colorectal cancer' as a studied disease/condition?", "Does the study investigate 'Crizotinib' as an intervention, treatment, exposure, or experimental factor for 'Colorectal cancer'?", "Does the study evaluate the therapeutic effectiveness or antitumor activity of 'Crizotinib' for 'Colorectal cancer' using clinically or biologically relevant efficacy outcomes?"]
    model_errors = error_index | criteria_num | error_description
0 | 1 | The model failed to distinguish between the study's primary subject (biomarkers) and the diseases listed as examples. It incorrectly assumed that mentioning CRC in a list of biomarker applications meant CRC was the studied condition, ignoring that the paper reviews biomarkers generally.
1 | 1 | The model incorrectly scored UNCERTAIN (0) instead of NO (-1). It failed to recognize that the complete absence of any mention o

INFO:root:fresh response in asynch: reasoning="The errors can be clustered into four distinct root causes based on the cognitive mechanism of the failure:\n1. **Confusing Background/Example Mentions with Study Focus**: The model incorrectly interprets a disease mentioned in a list, background, or as an example as the primary subject of the study. This applies to Criteria 1 and 2.\n2. **Misinterpreting Study Design/Methodology**: The model fails to distinguish between primary studies (patients/samples) and reviews/policy analyses, or between experimental/clinical investigations and in silico/computational predictions. This applies to Criteria 1 and 2.\n3. **Treating Explicit Exclusion or Absence as Ambiguity**: The model assigns 'Uncertain' when the text explicitly excludes the condition, limits the scope to another disease, or completely lacks the required information, which should result in a 'No'. This applies to Criteria 1, 2, and 3.\n4. **Ignoring Population Specificity in Efficacy

In [3]:

z,alldf = main.augment_samples(error_type_df,correct_df,
                        outs,
                        a[a==3].index.values[:1],
                        fin_condition,treatment)

INFO:root: passing 1 queries to llama.cpp...


http://localhost:8501/v1 /models/Qwen3.6-27B-UD-Q4_K_XL.gguf


INFO:root:fresh response in asynch: new_samples=[ScreeningSample(error_type_index=0, reasoning='Modified the abstract to mention Colorectal Cancer (CRC) only in the background context as an example of a disease where HGF/c-Met is relevant, while the actual study focuses exclusively on breast and lung cancer models. This tests if the model incorrectly identifies CRC as a studied population based on background mentions.', new_abstract='HGF/c-Met pathway is one of the mediators of sunitinib-induced tumor cell type-dependent metastasis. Recent studies in several tumor models, including colorectal cancer (CRC), indicated that treatment with angiogenic inhibitors may trigger induction of metastasis to other organs. Here we investigated modes of resistance and invasion in several tumor cell lines including 4T1 (breast) and H460 (lung) using sunitinib at doses comparable to clinically utilized regimen. In comparison with vehicle-treated tumors, sunitinib increased metastasis to lung in 4T1 tum

In [4]:
error_type_df.error_type.unique()

array(['Confusing background/example mentions with study focus',
       'Treating explicit exclusion or absence as ambiguity',
       'Misinterpreting study design or methodology type',
       'Ignoring population specificity in efficacy evaluation'],
      dtype=object)

In [5]:
correct_score,alldf = main.add_correct_scores(z,
                    fin_condition,treatment)

INFO:root: passing 1 queries to llama.cpp...


http://localhost:8501/v1 /models/Qwen3.6-27B-UD-Q4_K_XL.gguf


INFO:root:fresh response in asynch: new_samples=[SampleScore(error_type_index=0, reasoning='CRC is mentioned only in the background as an example of prior studies. The actual investigation focuses exclusively on breast (4T1) and lung (H460) models. This tests if the model incorrectly includes CRC based on background context.', new_ground_truth=[CriteriaScore(criteria_index=0, new_expert_rationale="CRC is mentioned only in the background ('Recent studies... including colorectal cancer'). The study explicitly investigates '4T1 (breast) and H460 (lung)' models. No CRC samples were studied.", new_expert_score=-1), CriteriaScore(criteria_index=1, new_expert_rationale='Since no CRC models were included in the study, Crizotinib was not investigated for CRC.', new_expert_score=-1), CriteriaScore(criteria_index=2, new_expert_rationale='No efficacy outcomes for Crizotinib in CRC are reported because CRC was not a studied population.', new_expert_score=-1)]), SampleScore(error_type_index=1, reaso

In [7]:
alldf

,error_type_index,reasoning,criteria_index,new_expert_rationale,new_expert_score
0,0,CRC is mentioned only in the background as an ...,0,CRC is mentioned only in the background ('Rece...,-1
1,0,CRC is mentioned only in the background as an ...,1,Since no CRC models were included in the study...,-1
2,0,CRC is mentioned only in the background as an ...,2,No efficacy outcomes for Crizotinib in CRC are...,-1
3,1,All mentions of CRC and the Colo205 cell line ...,0,"The abstract lists '4T1 (breast), H460 (lung) ...",-1
4,1,All mentions of CRC and the Colo205 cell line ...,1,Crizotinib was tested in the listed models (br...,-1
5,1,All mentions of CRC and the Colo205 cell line ...,2,No efficacy data for Crizotinib in CRC is prov...,-1
6,2,The study design is changed to 'in silico comp...,0,The study is an 'in silico computational scree...,-1
7,2,The study design is changed to 'in silico comp...,1,The investigation is computational ('predicted...,-1
8,2,The study design is changed to 'in silico comp...,2,Outcomes are based on 'computational models pr...,-1
9,3,CRC is part of a 'broad cohort of advanced sol...,0,The study investigates a 'broad cohort of adva...,1


## fixing types of errors

In [ ]:
df = pd.read_csv('common_errors.csv',sep=';')
chosen_cols = df.columns[~df.columns.isin(['контекст/уточнения'])]
df = df[chosen_cols].dropna(how='all')
df['Оценка_Эксперт']=df['Оценка_Эксперт'].astype(int)
df['Новая оценка эксперта'] = df[['Новая оценка эксперта','Тип_ошибки']].apply(lambda x: x['Новая оценка эксперта'] \
                                                                               if x['Тип_ошибки']!='In silico вместо фактического применения' else None,
                                                                               axis=1)
df['Оценка_Эксперт'] = df[['Оценка_Эксперт','Новая оценка эксперта']].apply(lambda x: int(x['Новая оценка эксперта']) \
                                                     if not pd.isna(x['Новая оценка эксперта']) else x['Оценка_Эксперт'],
                                                     axis=1)
df['Обоснование_Эксперт'] = df[['Обоснование_Эксперт','Новое обосновение эксперта']].apply(lambda x: x['Новое обосновение эксперта'] \
                                                     if not pd.isna(x['Новое обосновение эксперта']) else x['Обоснование_Эксперт'],
                                                     axis=1)
df['Почему_ошибка'] = df[['Почему_ошибка','Новое почему ошибка']].apply(lambda x: x['Новое почему ошибка'] \
                                                     if not pd.isna(x['Новое почему ошибка']) else x['Почему_ошибка'],
                                                     axis=1)
df['Тип_ошибки'] = df[['Новый тип ошибки','Тип_ошибки']].apply(lambda x: x['Новый тип ошибки'] if (not pd.isna(x['Новый тип ошибки'])) else x['Тип_ошибки'],
                                                         axis=1)

'In silico вместо фактического применения'
'The model mistakes a diagnostic/imaging tool for a therapeutic intervention.'

df.loc[df['Тип_ошибки'].isin(['In silico вместо фактического применения',
                         'The model mistakes a diagnostic/imaging tool for a therapeutic intervention.']),
       'Тип_ошибки'] = 'The model mistakes a diagnostic/imaging/prognostic tool for an investigated therapeutic intervention.'
df.loc[df['Тип_ошибки']=='The criteria is partially met', 'Тип_ошибки'] = 'The criteria is only partially met'
print(df.shape, df[df['Оценка_ЛЛМ']!=df['Оценка_Эксперт']].shape)
df = df[df['Оценка_ЛЛМ']!=df['Оценка_Эксперт']].iloc[:,:8]
#df.to_csv('common_errors_fixed.csv',index=False)
df.head(2)

(40, 12) (35, 12)


,PMID_статьи,Номер_критерия,Оценка_ЛЛМ,Обоснование_LLM,Оценка_Эксперт,Обоснование_Эксперт,Почему_ошибка,Тип_ошибки
0,25139705.0,1.0,1.0,The text explicitly lists 'colorectal cancer' ...,-1.0,"Это обзорная статья, обобщающая текущие достиж...",LLM приняла упоминание КРР в обзоре как фокус ...,The model confuses background keywords with th...
1,31399568.0,1.0,1.0,The text explicitly states that 'ROS1 gene rea...,-1.0,В абстракте крр упомянут как один из примеров ...,LLM интерпретировала КРР как изучаемое заболев...,The model confuses background keywords with th...


In [60]:
df['Тип_ошибки'].value_counts()

Тип_ошибки
The model confuses background keywords with the paper's focus                                            14
The model treats 'not mentioned' as 'possible'                                                           11
The model mistakes a diagnostic/imaging/prognostic tool for an investigated therapeutic intervention.     5
The criteria is only partially met                                                                        2
The model equates keyword co-occurrence with keyword interaction.                                         1
The model mistakes a proposed idea for a concept under study.                                             1
The model confuses a general topic with the potential inclusion of specific data.                         1
Name: count, dtype: int64

## translate

In [5]:
%pip uninstall -y googletrans 
%pip install googletrans-curl

Note: you may need to restart the kernel to use updated packages.


  Using cached googletrans_curl-4.1.0-py3-none-any.whl.metadata (9.1 kB)
  Using cached curl_cffi-0.16.3-cp310-abi3-win_amd64.whl.metadata (17 kB)
  Using cached cffi-2.1.1-cp310-cp310-win_amd64.whl.metadata (2.6 kB)
  Using cached pycparser-3.11-py3-none-any.whl.metadata (8.2 kB)
Using cached googletrans_curl-4.1.0-py3-none-any.whl (21 kB)
   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/2.0 MB ? eta -:--:--
   ---------------------------------------- 2.0/2.0 MB 8.5 MB/s  0:00:00

   ---------- ----------------------------- 1/4 [cffi]
   ---------- ----------------------------- 1/4 [cffi]
   ---------- ----------------------------- 1/4 [cffi]
   -------------------- ------------------- 2/4 [curl-cffi]
   -------------------- ------------------- 2/4 [curl-cffi]
   -------------------- ------------------- 2/4 [curl-cffi]
   -------------------- ------------------- 2/4 [curl-cffi]
   ------------------------------ ------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 26.0.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
from googletrans import SyncTranslator
import re
translator = SyncTranslator()
text=['В статье описывается клинический случай 19-летнего пациента с злокачественной перитонеальной мезотелиомой',
       'Кризотиниб применялся для лечения мезотелиомы',
       'Эффективность кризотиниба для крр не оценивается','Кризотиниб упоминается в абстракте, но в связи с нмрл и транслокациями ALK']
text = [re.sub(r'\b[нН][мМ][Рр][лЛ]\b','NSCLC', re.sub(r'\b[Кк][Рр][Рр]\b','CRC',i)) for i in text]
r = translator.translate([i for i in text], dest="en")
[i.text for i in r]#.text

['The article describes the clinical case of a 19-year-old patient with malignant peritoneal mesothelioma',
 'Crizotinib has been used to treat mesothelioma',
 'The effectiveness of crizotinib for CRC has not been assessed',
 'Crizotinib is mentioned in the abstract, but in association with NSCLC and ALK translocations']

In [ ]:
from googletrans import SyncTranslator
import re
translator = SyncTranslator()
text = (dfo.iloc[:,4:].values).reshape(-1)
text = [re.sub(r'\b[нН][мМ][Рр][лЛ]\b','NSCLC', re.sub(r'\b[Кк][Рр][Рр]\b','CRC',i)) for i in text]
r = translator.translate([i for i in text], dest="en")
[i.text for i in r]#.text

['The article describes the clinical case of a 19-year-old patient with malignant peritoneal mesothelioma',
 'Crizotinib has been used to treat mesothelioma',
 'The effectiveness of crizotinib for CRC has not been assessed.',
 '"16 colorectal cancer cases"',
 'The abstract states that patients received inhibitors of various signaling pathways, including ALK/ROS1. Although crizotinib is an ALK/ROS1 inhibitor, it is not explicitly named in the abstract. We cannot confirm that it was the specific drug used in patients (it could be ceritinib, alectinib, lorlatinib, etc.)',
 'The abstract provides general data for the cohort (4 patients with CRC had a response), but is not linked to a specific drug.',
 'This is a review article summarizing current advances in targeted therapy for solid tumors. It does not include original patients, models or samples',
 'Crizotinib is mentioned in the abstract, but in association with NSCLC',
 'Therapeutic efficacy for CRC has not been assessed',
 'The study

In [ ]:
dfo.iloc[:,4:] = np.array([i.text for i in r]).reshape(-1,3)
#dfo.to_csv('/content/drive/MyDrive/new_criteria_ana_en.csv',index=False)